<a href="https://colab.research.google.com/github/solivagvs/Stat-554/blob/main/sandorHW07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name: Bryan Sandor

Course: Stat 554

Title: Homework 7

# Preamble

**Summary**: This homework assists in the practice of multiple linear regression (MLR) and logistic models. The data used is about wine quality subject to the following variables:
- Input variables
    1. fixed acidity
    2. volatile acidity
    3. citric acid
    4. residual sugar
    5. chlorides
    6. free sulfur dioxide
    7. total sulfur dioxide
    8. density
    9. pH
    10. sulphates
    11. alcohol
- Output variable
    1. quality (a score between $0$ and $10$)

Now, instead of using the data to predict the quality, the target for fitting MLR models is `alcohol` and for logistic regression models we use the `type` as the response.

# Read In and Combine Data

The data was obtained from the UCI machine learning repository website https://archive.ics.uci.edu/. It includes two sets of data, one for red wine and the other for white wine.

In [1]:
import pandas as pd
import numpy as np

redwine = pd.read_csv("winequality-red.csv", delimiter = ";")
whiwine = pd.read_csv("winequality-white.csv", delimiter = ";")

Now we combine the two datasets into a single data set, first preserving the type of wine in an additional _numeric_ column, $0$ for red and $1$ for white.

In [2]:
redwine["type"] = 0 # Assign red wines a code value of 0
whiwine["type"] = 1 # Assign white wines a code value of 1

In [3]:
winedat = pd.concat([redwine, whiwine], ignore_index = True)

Now we show a small random sample of the data to show the results are preserved.

In [4]:
winedat.sample(10, random_state = 1982) # random sample of 10 cells, with a seed

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
923,6.8,0.410,0.31,8.8,0.084,26.0,45.0,0.998240,3.38,0.64,10.1,6,0
5558,5.6,0.185,0.19,7.1,0.048,36.0,110.0,0.994380,3.26,0.41,9.5,6,1
1905,7.2,0.430,0.24,6.7,0.058,40.0,163.0,0.995000,3.20,0.41,9.9,5,1
1780,7.1,0.340,0.15,1.2,0.053,61.0,183.0,0.993600,3.09,0.43,9.2,5,1
1520,6.5,0.530,0.06,2.0,0.063,29.0,44.0,0.994890,3.38,0.83,10.3,6,0
6145,6.5,0.200,0.31,2.1,0.033,32.0,95.0,0.989435,2.96,0.61,12.0,6,1
1324,6.7,0.460,0.24,1.7,0.077,18.0,34.0,0.994800,3.39,0.60,10.6,6,0
2349,7.2,0.290,0.40,7.6,0.024,56.0,177.0,0.992800,3.04,0.32,11.5,6,1
4798,6.8,0.210,0.40,6.3,0.032,40.0,121.0,0.992140,3.18,0.53,12.0,7,1
4219,6.5,0.280,0.28,20.4,0.041,40.0,144.0,1.000200,3.14,0.38,8.7,5,1


# Split the Data

Now we will split up the data into a
1. training set and a
2. test set.

We must use stratified sampling to ensure we have similar proportions of red and white wines across the two sets.

In [5]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso

X_train, X_test, y_train, y_test = train_test_split(
  winedat.drop("alcohol", axis = 1),
  winedat["alcohol"],
  test_size = 0.20,
  stratify = winedat["type"],
  random_state = 1982)

Now we verify the initial proportion of red and whites across the data sets are preserved.

In [6]:
print("Original dataset class distribution:")
print(winedat["type"].value_counts(normalize=True))
print("\nTraining set class distribution:")
print(X_train.value_counts("type", normalize=True))
print("\nTesting set class distribution:")
print(X_test.value_counts("type", normalize=True))

Original dataset class distribution:
type
1    0.753886
0    0.246114
Name: proportion, dtype: float64

Training set class distribution:
type
1    0.753896
0    0.246104
Name: proportion, dtype: float64

Testing set class distribution:
type
1    0.753846
0    0.246154
Name: proportion, dtype: float64


# Regression Task

We will now train several models, using the variable `alcohol` as the response.

## Train Models

### Multiple Linear Regression (MLR) Models

#### Standardization

First we standardize all the variables so they all have mean $0$ and standard deviation $1$.

In [7]:
means = X_train.apply(np.mean, axis = 0)
means

,0
fixed acidity,7.223629
volatile acidity,0.340819
citric acid,0.319225
residual sugar,5.451924
chlorides,0.056014
free sulfur dioxide,30.438426
total sulfur dioxide,115.671349
density,0.994704
pH,3.216421
sulphates,0.530471


In [8]:
stds = X_train.apply(np.std, axis = 0)
stds

,0
fixed acidity,1.303257
volatile acidity,0.165255
citric acid,0.145772
residual sugar,4.791132
chlorides,0.034533
free sulfur dioxide,17.879413
total sulfur dioxide,56.664252
density,0.003014
pH,0.160284
sulphates,0.148753


The following divides each entry by its category's standard deviation after subtracting the category's mean from it, standardizing the entries.

In [9]:
X_train = X_train.apply(lambda x: (x - np.mean(x)) / np.std(x), axis = 0)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4235,-0.018131,-0.610081,-0.337682,1.241476,-0.637487,0.534781,1.029373,1.003997,-0.102450,-0.608198,-0.919828,0.571351
4011,-0.708708,0.237096,-1.023684,0.698807,0.694578,1.094084,1.699990,1.103529,1.582061,0.870761,-2.053955,0.571351
3617,-0.171592,-0.852132,0.142520,-0.699610,-0.203118,-1.031266,0.358756,-0.830710,0.209497,-0.473747,1.348426,0.571351
3783,-0.631977,-0.065467,-0.543483,0.907526,-0.434781,-0.080452,-0.117735,0.297319,0.459054,-0.406521,0.214299,0.571351
502,2.437255,0.600173,2.817930,0.229189,0.520830,0.422921,-0.700112,1.425349,-0.289617,2.148043,1.348426,-1.750237
...,...,...,...,...,...,...,...,...,...,...,...,...
4968,-0.171592,-0.731107,-0.269081,1.074501,-0.492697,0.087339,0.023448,-0.382816,-1.038289,0.131282,1.348426,0.571351
1007,1.439755,-0.247005,0.142520,-0.720482,0.231251,-1.031266,-1.600151,0.151339,0.271886,2.080817,1.348426,-1.750237
5244,-1.015631,-0.973158,-0.269081,-0.073453,-0.492697,1.094084,0.411700,-1.013186,-0.352007,-0.608198,0.214299,0.571351
4919,0.212062,-0.307518,0.279721,2.138968,-0.174160,-0.080452,0.146982,0.695447,-0.975899,0.064056,-0.919828,0.571351


We verify the standardization:

In [13]:
X_train.apply(np.mean, axis = 0)

,0
fixed acidity,-3.059148e-16
volatile acidity,8.203303e-18
citric acid,1.415070e-16
residual sugar,-1.189479e-16
chlorides,-7.519694e-18
free sulfur dioxide,-8.203303e-18
total sulfur dioxide,-2.529352e-17
density,2.134602e-14
pH,3.442995e-15
sulphates,-2.378958e-16


Numerically, the means are all zero.

In [14]:
X_train.apply(np.std, axis = 0)

,0
fixed acidity,1.0
volatile acidity,1.0
citric acid,1.0
residual sugar,1.0
chlorides,1.0
free sulfur dioxide,1.0
total sulfur dioxide,1.0
density,1.0
pH,1.0
sulphates,1.0
